In [16]:
from pymilvus import (
    connections,
    FieldSchema, CollectionSchema, DataType,
    Collection
)
from pymilvus import connections




In [17]:
from pymilvus import connections

# Recupera conexão existente
conn = connections.get_connection("default")
print(conn)  # Deve mostrar detalhes da conexão


AttributeError: 'Connections' object has no attribute 'get_connection'

In [18]:
from pymilvus import connections, Collection, CollectionSchema, FieldSchema, DataType
from sentence_transformers import SentenceTransformer
import uuid, requests
from PIL import Image, ImageEnhance
import cv2, numpy as np, easyocr
from io import BytesIO
from transformers import CLIPProcessor, CLIPModel

# ---- Conexão Milvus ----
connections.disconnect("default")
connections.connect("default", host="localhost", port="19530")

# Definição do schema
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, max_length=36, is_primary=True, auto_id=False),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),  # depende do modelo
    FieldSchema(name="url", dtype=DataType.VARCHAR, max_length=500),
    FieldSchema(name="category", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="titles", dtype=DataType.VARCHAR, max_length=500),
    FieldSchema(name="texts", dtype=DataType.VARCHAR, max_length=2000),
]
schema = CollectionSchema(fields, description="Armazenamento de imagens processadas")
collection = Collection("image_descriptions", schema)

# Embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


def classify_image(image_path: str) -> str:
    labels = ["Mapa", "Gráfico", "Diagrama", "Tabela", "Outro"]

    if image_path.startswith("http"):
        response = requests.get(image_path)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_path).convert("RGB")

    inputs = clip_processor(text=labels, images=image, return_tensors="pt", padding=True)
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)
    category = labels[probs.argmax().item()]
    return category


def extract_text_structure(image_path: str) -> dict:
    if image_path.startswith("http"):
        response = requests.get(image_path)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_path).convert("RGB")

    image = image.convert("L")
    image = ImageEnhance.Contrast(image).enhance(2)
    image = ImageEnhance.Sharpness(image).enhance(2)

    image_np = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    reader = easyocr.Reader(["pt"])
    result = reader.readtext(image_np)

    titles, texts = [], []
    for (bbox, text, prob) in result:
        if prob > 0.5:
            if bbox[1][1] < 50:
                titles.append(text.strip())
            else:
                texts.append(text.strip())

    return {"titles": titles, "texts": texts}


def process_and_store(image_url: str):
    try:
        category = classify_image(image_url)
        extracted = extract_text_structure(image_url)

        description = f"{category}: " + " ".join(extracted["titles"] + extracted["texts"])
        embedding = embedding_model.encode(description).tolist()

        doc_id = str(uuid.uuid4())

        # Inserção no Milvus
        collection.insert([[
            doc_id,
            embedding,
            image_url,
            category,
            " ".join(extracted["titles"]),
            " ".join(extracted["texts"])
        ]])

        collection.flush()
        print(f"[OK] Imagem armazenada no Milvus com ID {doc_id}")

    except Exception as e:
        print(f"Erro ao processar/armazenar imagem: {e}")


# Teste
process_and_store("https://smastr16.blob.core.windows.net/igeo/2012/03/mapa_aguas_subterraneas.jpg")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
C:\Users\vinic\Documents\GitHub\Database-Chroma-RAG-Project\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Erro ao processar/armazenar imagem: <DataNotMatchException: (code=1, message=The data doesn't match with schema fields, expect 6 list, got 1)>


In [ ]:
import uuid

def store_image_milvus(image_url, category, titles, texts, embedding):
    image_id = str(uuid.uuid4())

    data = [
        [image_id],
        [embedding],
        [image_url],
        [category],
        ["; ".join(titles)],
        ["; ".join(texts)]
    ]

    image_collection.insert(data)
    print(f"[OK] Imagem {image_id} salva no Milvus.")


In [ ]:
def search_images(query: str, top_k: int = 5):
    query_embedding = embedding_model.encode(query).tolist()
    image_collection.load()
    results = image_collection.search(
        data=[query_embedding],
        anns_field="embedding",
        param={"metric_type": "IP", "params": {"nprobe": 10}},
        limit=top_k,
        output_fields=["url", "category", "titles", "texts"]
    )
    for r in results[0]:
        print(f"Score: {r.score:.4f}")
        print(f"URL: {r.entity.get('url')}")
        print(f"Categoria: {r.entity.get('category')}")
        print(f"Títulos: {r.entity.get('titles')}")


In [15]:
from pymilvus import connections, utility

# Garante que está conectado (desconectando antes, só por segurança)
connections.disconnect("default")
connections.connect("default", host="localhost", port="19530")

# Nome da coleção
collection_name = "image_descriptions"

if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)
    print(f"Coleção '{collection_name}' excluída com sucesso.")
else:
    print(f"Coleção '{collection_name}' não existe.")


Coleção 'image_descriptions' não existe.
